In [2]:
"""实验5：拉普拉斯金字塔融合 —— 算法版（pyrDown / pyrUp 全部从零实现）"""
import tkinter as tk
from tkinter import ttk, filedialog
import numpy as np
import cv2  # 仅 I/O
import matplotlib
matplotlib.use('TkAgg')
matplotlib.rcParams['font.sans-serif']=['SimHei','Microsoft YaHei','Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus']=False
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk

# ============================================================
#                       算法核心部分（实现）
# ============================================================

def pad_reflect(img, r):
    """单通道反射填充"""
    H,W=img.shape
    out=np.zeros((H+2*r, W+2*r),dtype=np.float32)
    out[r:r+H, r:r+W]=img
    out[:r, r:r+W]=img[r:0:-1,:]
    out[r+H:, r:r+W]=img[H-2:H-2-r:-1,:]
    pW = r
    out[:, :pW] = out[:, 2*r:r:-1]
    # out[:, :pW := r]=out[:, 2*r:r:-1]
    out[:, r+W:]=out[:, r+W-2:W-2:-1]
    return out

def make_gaussian_kernel(k, sigma):
    """二维高斯核"""
    r=k//2
    kernel=np.zeros((k,k),dtype=np.float32)
    for i in range(k):
        for j in range(k):
            x=i-r; y=j-r
            kernel[i,j]=np.exp(-(x*x+y*y)/(2*sigma*sigma))
    return kernel/kernel.sum()


def conv2d_single(img, kernel):
    """单通道卷积（思路同前）"""
    H,W=img.shape; kH,kW=kernel.shape
    pH,pW=kH//2, kW//2
    padded=pad_reflect(img.astype(np.float32), pH)
    out=np.zeros((H,W),dtype=np.float32)
    for m in range(kH):
        for n in range(kW):
            out += kernel[m,n]*padded[m:m+H, n:n+W]
    return out


def conv2d_color(img, kernel):
    """
    多通道卷积：对每个颜色通道独立卷积
    """
    if img.ndim == 2:                       # 灰度图直接走单通道版本
        return conv2d_single(img, kernel)
    out = np.zeros_like(img, dtype=np.float32)
    for c in range(img.shape[2]):           # 对 R/G/B 三个通道
        out[..., c] = conv2d_single(img[..., c], kernel)
    return out


def pyr_down(img, kernel):
    """
    金字塔下采样   pyrDown
    -------------------------------------------------
    步骤：
      1) 用高斯核做低通滤波（防止下采样产生混叠/锯齿）
      2) 每隔一行/一列丢弃像素（也就是 [::2, ::2]）
    结果尺寸约为原来的 1/2 × 1/2。
    """
    blurred = conv2d_color(img, kernel)     # 1) 低通滤波
    return blurred[::2, ::2]                # 2) 隔行隔列采样


def pyr_up(img, kernel, target_shape):
    """
    金字塔上采样   pyrUp
    -------------------------------------------------
    步骤：
      1) 在每行/每列之间"插 0"，尺寸变为约 2H × 2W
      2) 用 4×kernel 做卷积，把 0 值补成相邻像素的加权平均
         （乘 4 是为了补回插 0 时损失的能量）
    最终用 target_shape 裁剪/对齐到目标层尺寸。
    """
    H, W = img.shape[:2]
    if img.ndim == 3:
        up = np.zeros((H*2, W*2, 3), dtype=np.float32)
    else:
        up = np.zeros((H*2, W*2), dtype=np.float32)
    up[::2, ::2] = img                      # 1) 把原像素放回偶数行/列
    out = conv2d_color(up, 4 * kernel)      # 2) 卷积补出"插值"
    return out[:target_shape[0], :target_shape[1]]


def gaussian_pyramid(img, levels, kernel):
    """
    构造高斯金字塔：[G0, G1, G2, ...]
    G0 = 原图，G_{i+1} = pyr_down(G_i)
    """
    pyr = [img.astype(np.float32)]
    for _ in range(levels):
        pyr.append(pyr_down(pyr[-1], kernel))
    return pyr


def laplacian_pyramid(img, levels, kernel):
    """
    构造拉普拉斯金字塔：[L0, L1, ..., L_{n-1}, G_n]
    
    定义：L_i = G_i - pyr_up(G_{i+1})
    每层 L_i 保存了"该层比下一层多出来的高频细节"，
    最后一层保留高斯金字塔的最低分辨率图（顶端）。
    
    重建公式：G_i = L_i + pyr_up(G_{i+1})
    所以可以从最顶端逐层向上加回，得到原图（精确重建）。
    """
    G = gaussian_pyramid(img, levels, kernel)
    L = []
    for i in range(levels):
        # 把更小一层 G[i+1] 升回到 G[i] 的尺寸再相减
        up = pyr_up(G[i+1], kernel, G[i].shape)
        L.append(G[i] - up)
    L.append(G[-1])         # 顶端那张低分辨率图直接保留
    return L


def pyramid_blend(A, B, mask, levels, kernel):
    """
    拉普拉斯金字塔图像融合
    -------------------------------------------------
    思路：直接按 mask 拼图会有明显接缝；
          通过在每个频段（每一层金字塔）独立按"该层 mask"做加权，
          再逐层重建，过渡区会随着层级越深越宽，最终接缝消失。
    
    公式：LS_i = GM_i · LA_i + (1 - GM_i) · LB_i
         其中 LA / LB 是 A、B 的拉普拉斯金字塔，
              GM   是 mask 的高斯金字塔
    """
    LA = laplacian_pyramid(A, levels, kernel)       # A 的拉普拉斯金字塔
    LB = laplacian_pyramid(B, levels, kernel)       # B 的拉普拉斯金字塔
    GM = gaussian_pyramid(mask, levels, kernel)     # mask 的高斯金字塔
    # 1) 每层各自加权融合
    LS = [gm*la + (1-gm)*lb for la, lb, gm in zip(LA, LB, GM)]
    # 2) 从最顶端开始逐层向上重建
    out = LS[-1]
    for i in range(levels-1, -1, -1):
        out = pyr_up(out, kernel, LS[i].shape) + LS[i]
    return np.clip(out, 0, 255).astype(np.uint8)


def direct_blend(A, B, mask):
    """直接加权拼图（用作对比，会出现明显接缝）"""
    return np.clip(mask*A + (1-mask)*B, 0, 255).astype(np.uint8)


def make_mask(size, mode, feather):
    """
    生成融合 mask
    -------------------------------------------------
    mode    : '左右' / '圆形' / '对角'
    feather : 过渡带宽（像素），>0 时会用线性渐变软化边缘
    """
    m = np.zeros((size, size, 3), dtype=np.float32)
    if mode == '左右':
        m[:, :size//2] = 1
        if feather > 0:
            for x in range(size//2 - feather, size//2 + feather):
                if 0 <= x < size:
                    # 在 [c-f, c+f] 内做线性渐变
                    t = (size//2 + feather - x) / (2 * feather)
                    m[:, x] = np.clip(t, 0, 1)
    elif mode == '圆形':
        cy = cx = size // 2
        R = size // 3
        for y in range(size):
            for x in range(size):
                d = np.sqrt((y-cy)**2 + (x-cx)**2)
                if feather > 0:
                    # 在半径附近平滑过渡
                    v = np.clip((R + feather - d) / (2 * feather), 0, 1)
                else:
                    v = 1.0 if d <= R else 0.0
                m[y, x] = v
    else:  # 对角
        for y in range(size):
            for x in range(size):
                # x+y < size 的一侧为 A，反之为 B
                diff = size - (x + y)
                if feather > 0:
                    v = np.clip((diff + feather) / (2 * feather), 0, 1)
                else:
                    v = 1.0 if diff > 0 else 0.0
                m[y, x] = v
    return m

# ============================================================
#                       GUI
# ============================================================
BG_DARK='#1e1e2e'; BG_PANEL='#252537'; BG_LIGHT='#f7f7fa'
FG_TEXT='#e4e4ef'; FG_MUTED='#9090a8'; ACCENT='#7c9cff'; ACCENT_2='#ff8a8a'

def make_demo_AB():
    A=np.zeros((128,128,3),np.uint8); A[:]=(40,40,200)
    cv2.circle(A,(64,64),34,(60,60,250),-1)
    B=np.zeros((128,128,3),np.uint8); B[:]=(60,180,60)
    cv2.rectangle(B,(30,30),(98,98),(100,230,100),-1)
    return A,B

class LabeledSlider(tk.Frame):
    def __init__(self,master,text,frm,to,init,cb,fmt='{:.2f}',res=0.01):
        super().__init__(master,bg=BG_PANEL); self.cb=cb; self.fmt=fmt; self.res=res
        top=tk.Frame(self,bg=BG_PANEL); top.pack(fill='x',pady=(8,0))
        tk.Label(top,text=text,bg=BG_PANEL,fg=FG_TEXT,font=('Segoe UI',10,'bold')).pack(side='left')
        self.val_lbl=tk.Label(top,text=fmt.format(init),bg=BG_PANEL,fg=ACCENT,
                              font=('Consolas',10,'bold')); self.val_lbl.pack(side='right')
        self.var=tk.DoubleVar(value=init)
        ttk.Scale(self,from_=frm,to=to,variable=self.var,orient='horizontal',
                  command=self._on).pack(fill='x',pady=(2,6))
    def _on(self,_):
        v=self.var.get()
        if self.res>=1: v=round(v)
        self.val_lbl.config(text=self.fmt.format(v)); self.cb()
    def get(self):
        v=self.var.get(); return round(v) if self.res>=1 else v

class App:
    def __init__(self,root):
        self.root=root
        root.title('实验5 · 拉普拉斯金字塔融合 — 算法版')
        root.geometry('1380x900'); root.configure(bg=BG_DARK)
        self.A,self.B=make_demo_AB()
        self._style(); self._ui(); self._update()

    def _style(self):
        st=ttk.Style(); st.theme_use('clam')
        st.configure('TScale',background=BG_PANEL,troughcolor='#3a3a50',
                     bordercolor=BG_PANEL,lightcolor=ACCENT,darkcolor=ACCENT)
        st.configure('TCombobox',fieldbackground='#3a3a50',background='#3a3a50',
                     foreground=FG_TEXT,arrowcolor=FG_TEXT)

    def _ui(self):
        side=tk.Frame(self.root,bg=BG_PANEL,width=300); side.pack(side='left',fill='y'); side.pack_propagate(False)
        tk.Label(side,text='⚙  参数控制',bg=BG_PANEL,fg=FG_TEXT,
                 font=('Segoe UI',14,'bold')).pack(anchor='w',padx=18,pady=(20,6))
        tk.Frame(side,bg=ACCENT,height=2).pack(fill='x',padx=18)

        wrap=tk.Frame(side,bg=BG_PANEL); wrap.pack(fill='x',padx=18,pady=(14,0))
        self.lv=LabeledSlider(wrap,'金字塔层数',1,4,3,self._update,'{:.0f}',1); self.lv.pack(fill='x')
        self.ksig=LabeledSlider(wrap,'高斯核 σ',0.5,2.5,1.0,self._update,'{:.2f}',0.01); self.ksig.pack(fill='x')
        self.feather=LabeledSlider(wrap,'mask 过渡(px)',0,20,0,self._update,'{:.0f}',1); self.feather.pack(fill='x')

        b=tk.Frame(side,bg=BG_PANEL); b.pack(fill='x',padx=18,pady=(8,4))
        tk.Label(b,text='Mask 类型',bg=BG_PANEL,fg=FG_TEXT,font=('Segoe UI',10,'bold')).pack(anchor='w')
        self.mode=tk.StringVar(value='左右')
        ttk.Combobox(b,textvariable=self.mode,state='readonly',
                     values=['左右','圆形','对角']).pack(fill='x',pady=(4,4))
        self.mode.trace_add('write',lambda *a: self._update())

        tk.Frame(side,bg=BG_PANEL,height=10).pack()
        self._btn(side,'📁  打开图 A',lambda: self._open('A'))
        self._btn(side,'📁  打开图 B',lambda: self._open('B'))
        self._btn(side,'💾  保存结果',self._save)
        self._btn(side,'🔄  重置参数',self._reset, ACCENT_2)

        info=tk.Frame(side,bg='#2c2c40'); info.pack(side='bottom',fill='x',padx=12,pady=12)
        self.info=tk.Label(info,text='',bg='#2c2c40',fg=FG_MUTED,font=('Consolas',9),justify='left',anchor='w')
        self.info.pack(fill='x',padx=10,pady=10)

        main=tk.Frame(self.root,bg=BG_LIGHT); main.pack(side='right',fill='both',expand=True)
        self.fig=plt.Figure(figsize=(13,9),facecolor=BG_LIGHT)
        self.canvas=FigureCanvasTkAgg(self.fig,master=main)
        self.canvas.get_tk_widget().pack(fill='both',expand=True,padx=8,pady=8)
        tb=NavigationToolbar2Tk(self.canvas,main); tb.update(); tb.configure(bg=BG_LIGHT)

    def _btn(self,p,t,c,col=ACCENT):
        tk.Button(p,text=t,command=c,bg=col,fg='white',activebackground='#5a7ae0',
                  relief='flat',font=('Segoe UI',10,'bold'),cursor='hand2',pady=8
                  ).pack(fill='x',padx=18,pady=4)

    def _open(self,which):
        p=filedialog.askopenfilename(filetypes=[('Image','*.png *.jpg *.jpeg *.bmp')])
        if p:
            im=cv2.imread(p)
            if im is not None:
                im=cv2.resize(im,(128,128))
                if which=='A': self.A=im
                else: self.B=im
                self._update()

    def _save(self):
        p=filedialog.asksaveasfilename(defaultextension='.png')
        if p: cv2.imwrite(p,self.result)

    def _reset(self):
        for w,t in [(self.lv,3),(self.ksig,1.0),(self.feather,0)]:
            w.var.set(t); w.val_lbl.config(text=w.fmt.format(t))
        self.mode.set('左右'); self._update()

    def _update(self):
        levels=int(self.lv.get())
        ksig=self.ksig.get(); feather=int(self.feather.get())
        size=128
        a=cv2.resize(self.A,(size,size)); b=cv2.resize(self.B,(size,size))
        kernel=make_gaussian_kernel(5, ksig)        # 5×5 高斯核（σ 可调）
        mask=make_mask(size, self.mode.get(), feather)
        self.result=pyramid_blend(a, b, mask, levels, kernel)
        dir_=direct_blend(a, b, mask)
        GM=gaussian_pyramid(mask, levels, kernel)

        self.fig.clear()
        gs=self.fig.add_gridspec(3,3,hspace=0.42,wspace=0.25,height_ratios=[1,1,0.75],
                                 left=0.04,right=0.98,top=0.96,bottom=0.07)
        ax_a=self.fig.add_subplot(gs[0,0]); ax_b=self.fig.add_subplot(gs[0,1])
        ax_m=self.fig.add_subplot(gs[0,2]); ax_d=self.fig.add_subplot(gs[1,0])
        ax_p=self.fig.add_subplot(gs[1,1]); ax_r=self.fig.add_subplot(gs[1,2])
        ax_curve=self.fig.add_subplot(gs[2,:])

        ax_a.imshow(cv2.cvtColor(a,cv2.COLOR_BGR2RGB)); ax_a.set_title('图 A',fontsize=11,fontweight='bold'); ax_a.axis('off')
        ax_b.imshow(cv2.cvtColor(b,cv2.COLOR_BGR2RGB)); ax_b.set_title('图 B',fontsize=11,fontweight='bold'); ax_b.axis('off')
        ax_m.imshow(mask); ax_m.set_title(f'Mask (feather={feather})',fontsize=11,fontweight='bold'); ax_m.axis('off')
        ax_d.imshow(cv2.cvtColor(dir_,cv2.COLOR_BGR2RGB))
        ax_d.set_title('直接加权 (有缝)',fontsize=11,fontweight='bold'); ax_d.axis('off')
        ax_p.imshow(cv2.cvtColor(self.result,cv2.COLOR_BGR2RGB))
        ax_p.set_title(f'金字塔融合 (层数={levels})',fontsize=11,fontweight='bold'); ax_p.axis('off')
        diff=np.abs(self.result.astype(int)-dir_.astype(int)).astype(np.uint8)
        ax_r.imshow(diff); ax_r.set_title('|金字塔 − 直接|',fontsize=11,fontweight='bold'); ax_r.axis('off')

        try:
            cmap=matplotlib.colormaps.get_cmap('viridis').resampled(len(GM))
        except Exception:
            cmap=matplotlib.cm.get_cmap('viridis',len(GM))
        for p in range(len(GM)):
            gm=GM[p][...,0]
            gm_up=cv2.resize(gm,(size,size))
            ax_curve.plot(np.arange(size),gm_up[size//2,:],
                          lw=2.2,alpha=0.88,color=cmap(p),label=f'level {p}')
        ax_curve.set_title('Mask 中心行在金字塔各层的过渡曲线（层级越高，过渡越平滑）',
                           fontsize=11,fontweight='bold')
        ax_curve.set_xlabel('x'); ax_curve.set_ylabel('权重')
        ax_curve.set_ylim(-0.05,1.1); ax_curve.legend(fontsize=8,ncol=min(len(GM),7),loc='upper right')
        ax_curve.grid(alpha=0.3)

        self.canvas.draw()
        self.info.config(text=f'层数: {levels}\n核 σ: {ksig:.2f}\nMask: {self.mode.get()}\n'
                              f'feather: {feather} px\n图像尺寸: {size}×{size}')

if __name__=='__main__':
    root=tk.Tk(); App(root); root.mainloop()

C:\Users\Albert\AppData\Local\Temp\ipykernel_25772\1267731300.py:346: UserWarning: Glyph 8722 (\N{MINUS SIGN}) missing from font(s) SimHei.
  self.canvas.draw()
C:\anaconda3\Lib\tkinter\__init__.py:862: UserWarning: Glyph 8722 (\N{MINUS SIGN}) missing from font(s) SimHei.
  func(*args)
